# Creating transaction bundles

This notebook groups several FHIR resources into a transaction bundle and sends them to the server in a single request. It assumes a FHIR server with data already loaded, as set up in [Loading FHIR data](3.1-load-fhir-data.ipynb), and uses `fhir.resources` as in [Creating FHIR resources](3.3-create-fhir-resources.ipynb).


In [1]:
import requests
from requests.auth import HTTPBasicAuth

## Credentials
username = "SuperUser"
password = "SYS"

## FHIR server location
baseURL = "http://localhost:52773/fhir/r4/"
headers ={    "Content-Type": "application/fhir+json"}


The bundle needs an existing patient to refer to. Here we use the first patient on the FHIR server:


In [2]:
res = requests.get(baseURL + "Patient?_count=1",
                   headers={"Accept": "application/fhir+json"},
                   auth=HTTPBasicAuth(username, password))
patient_id = res.json()["entry"][0]["resource"]["id"]
patient_ref = f"Patient/{patient_id}"
print(patient_ref)


Patient/13


## Working with multiple resources: **Bundles**

A bundle is a collection of FHIR resources. Bundles are a fundamental part of FHIR, as individual resources are designed to be small modular parts of larger stories. The bundle is the larger story. 

Each bundle must have a type component, with the type defining the purpose or provenence of the bundle. For example, if you query a FHIR Server for data, the bundle you recieve will have `type=searchset`. There are 10 possible types of bundles defined in [the HL7 FHIR specification](https://www.hl7.org/fhir/valueset-bundle-type.html). 

These include: 
|Bundle Type| Usage |
|-|-|
|message| A message between healthcare systems |
|searchset| The results from a query|
|document| A document containing relevant resources|
|collection| A set of resources packaged fo easy distribution|
|batch| A set of independent actions to be sent to a server|


However, the most common bundle generated by FHIR Clients is the `transaction` bundle. 

### Transaction bundles

Groups of FHIR resources can be bundled together into transactions. Transactions are a group of resources that are sent to a FHIR server combined with instructions about their processing. These resources are processed together, meaning if one of the resources errors, the rest of the bundle is not saved. To send resources in a way that processes the resources individually without fear of failure, batches can be used.

Transactions are used for grouped operations. For example, when a Patient has an X-Ray, a transaction bundle may be created with resources like an ImagingStudy resource, a DiagnositcReport and one or more Observation resources.    


#### Creating a Transaction bundle

Let's say, our patient goes to a Physician and our system needs to record this across two resources, an `Encounter` resource, and an `Observation` resource. We want to process these resources as a bundle.

When creating bundles, special attention should be given to references between resources. We can refer to existing resources in the same way we have seen before, by using the existing ID in the database. However, new resources don't yet have an ID in the database. Therefore, references should be created with unique placeholder values (using uuids)

Let's start by building the individual resources:


In [3]:
import uuid
from fhir.resources.R4B.encounter import Encounter
from fhir.resources.R4B.observation import Observation

encounter_ref = f"urn:uuid:{uuid.uuid4()}"
observation_ref = f"urn:uuid:{uuid.uuid4()}"

encounter = Encounter(
    status="finished",
    class_fhir={
        "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
        "code": "AMB",
        "display": "ambulatory",
    },
    subject={"reference": patient_ref},          # existing patient, real ID
    period={"start": "2026-10-14T09:00:00Z", "end": "2026-10-14T09:30:00Z"},
)

# Build the Observation (a heart rate taken during that visit)
observation = Observation(
    status="final",
    code={"coding": [{
        "system": "http://loinc.org",
        "code": "8867-4",
        "display": "Heart rate",
    }]},
    subject={"reference": patient_ref},          # existing patient, real ID
    encounter={"reference": encounter_ref},      # new encounter, placeholder
    effectiveDateTime="2026-10-14T09:10:00Z",
    valueQuantity={
        "value": 25, # Superman has a low resting heart-rate! 
        "unit": "beats/minute",
        "system": "http://unitsofmeasure.org",
        "code": "/min",
    },
)

In the above, we can see that the Observation currently refers to the `encounter_ref`, but this has not yet been given to the encounter. This reference is instead used when creating the bundle. 

To create a transaction bundle, we create a `Bundle` object, give a type ("transaction"), then include each bundle entry as a `BundleEntry` object, i.e.

```python
from fhir.resources.R4B.bundle import Bundle, BundleEntry

bundle = Bundle(
    type="transaction",
    entry=[
        BundleEntry(...),
        BundleEntry(...),
        BundleEntry(...)
        ]
    )
```

Inside the bundle reference, we require: 
- fullUrl (the reference uuid created above)
- resource (the resource object)
- request (detailed below)

The request can be created with `BundleEntryRequest`. This requires the Method and the FHIR endpoint (i.e. FHIR resource type). Method can be `PUT`, `POST`, `DELETE`, these are standard HTTP methods, where `POST` means "create a new value" and `PUT` means "insert new or replace existing". 

```python
from fhir.resources.R4B.bundle import BundleEntry, BundleEntryRequest

patient = Patient(....)

# Create a new patient 
entry = BundleEntry(
    fullUrl = f"urn:uuid:{uuid.uuid4()}", 
    resource = patient, # Resource object 
    request = BundleEntryRequest(   # Request details
        method = "POST", # Create a new 
        url = "Patient"  # Patient resource
       )
)

## Or replace an existing patient (avoid duplicates) 

entry = BundleEntry(
    fullUrl = f"urn:uuid:{uuid.uuid4()}", 
    resource = patient, # Resource object 
    request = BundleEntryRequest(
        method="PUT", # Replace or create new if it doesn't exist
        url="Patient?identifier=http://hospital.org/mrn|12345", # Combine with search for existing
    )
)
```
Conditional values can also be added to the `BundleEntryRequest` e.g. a POST request with `ifNoneExists` to ensure no duplicates are created. 

Let's see this in action with the resources we created above: 


In [4]:
from fhir.resources.R4B.bundle import Bundle, BundleEntry, BundleEntryRequest

# Assemble the transaction: two entries, three resources involved

bundle = Bundle(
    type="transaction",
    entry=[
        BundleEntry( 
            fullUrl=encounter_ref,
            resource=encounter,
            request=BundleEntryRequest(method="POST", url="Encounter"),
        ),
        BundleEntry(
            fullUrl=observation_ref,
            resource=observation,
            request=BundleEntryRequest(method="POST", url="Observation"),
        ),
    ],
)

## Inspect the resulting bundle
print(bundle.model_dump_json(indent=1))

{
 "resourceType": "Bundle",
 "type": "transaction",
 "entry": [
  {
   "fullUrl": "urn:uuid:3d159b43-218a-475d-adc4-79160ef4e3e6",
   "resource": {
    "resourceType": "Encounter",
    "status": "finished",
    "class": {
     "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
     "code": "AMB",
     "display": "ambulatory"
    },
    "subject": {
     "reference": "Patient/13"
    },
    "period": {
     "start": "2026-10-14T09:00:00Z",
     "end": "2026-10-14T09:30:00Z"
    }
   },
   "request": {
    "method": "POST",
    "url": "Encounter"
   }
  },
  {
   "fullUrl": "urn:uuid:f6df9dd3-f3c4-446b-bbd8-ebf8ff9ee551",
   "resource": {
    "resourceType": "Observation",
    "status": "final",
    "code": {
     "coding": [
      {
       "system": "http://loinc.org",
       "code": "8867-4",
       "display": "Heart rate"
      }
     ]
    },
    "subject": {
     "reference": "Patient/13"
    },
    "encounter": {
     "reference": "urn:uuid:3d159b43-218a-475d-adc4-79160

Let's try adding it to the FHIR server. Unlike above where we have been sending data to specific endpoints, here the data should be sent to the base FHIR server URL, as each resources has the destination endpoint individually labelled 

In [5]:
res = requests.post(baseURL, 
                    data=bundle.model_dump_json(), ## Give a JSON string as data - giving it a Python dict may cause errors because of the datetime object.
                    headers= headers, ## We are defining the content type (FHIR+JSON) in the header
                    auth=HTTPBasicAuth(username, password))
print(res)

<Response [200]>


We have a 200 response! That's a good sign the data has been accepted. Let's look at the response:

In [6]:
import json # used to pretty-print the response

print(json.dumps(res.json(), indent=1))

{
 "resourceType": "Bundle",
 "id": "c3af1d8c-c7c1-11f1-95ee-6a5d4b88ed41",
 "type": "transaction-response",
 "timestamp": "2026-10-14T11:24:05Z",
 "entry": [
  {
   "fullUrl": "urn:uuid:3d159b43-218a-475d-adc4-79160ef4e3e6",
   "response": {
    "status": "201",
    "location": "http://localhost:52773/fhir/r4/Encounter/2290",
    "etag": "W/\"1\"",
    "lastModified": "2026-10-14T11:24:05Z"
   }
  },
  {
   "fullUrl": "urn:uuid:f6df9dd3-f3c4-446b-bbd8-ebf8ff9ee551",
   "response": {
    "status": "201",
    "location": "http://localhost:52773/fhir/r4/Observation/2291",
    "etag": "W/\"1\"",
    "lastModified": "2026-10-14T11:24:05Z"
   }
  }
 ]
}


The response is another bundle - this time a `transaction-response`. Each entry has an individual outcome, which includes the location that it is saved to, with the final number being the database ID of the new resources. 

We can see the generated observation resource with the following:

In [7]:
obs_url = res.json()["entry"][1]["response"]["location"]
print(obs_url)

obs_res = requests.get(obs_url, 
                       headers =headers, 
                       auth = HTTPBasicAuth(username, password)
                      )

print(json.dumps(obs_res.json(), indent=1))

http://localhost:52773/fhir/r4/Observation/2291
{
 "resourceType": "Observation",
 "status": "final",
 "code": {
  "coding": [
   {
    "system": "http://loinc.org",
    "code": "8867-4",
    "display": "Heart rate"
   }
  ]
 },
 "subject": {
  "reference": "Patient/13"
 },
 "encounter": {
  "reference": "Encounter/2290"
 },
 "effectiveDateTime": "2026-10-14T09:10:00Z",
 "valueQuantity": {
  "value": 25,
  "unit": "beats/minute",
  "system": "http://unitsofmeasure.org",
  "code": "/min"
 },
 "id": "2291",
 "meta": {
  "lastUpdated": "2026-10-14T11:24:05Z",
  "versionId": "1"
 }
}


So the observation data is in the dataset, and you can see that the "encounter" being referenced has a true database ID. This ID should be the same as the number at the end of the location of the Encounter entry in the response printed above.

